# Notebook 05 — Orientação a Objetos (POO)

Até agora meu sistema usava **funções soltas** e **dicionários** para representar
cada aluno. Funciona, mas os dados (o dicionário) ficam separados das funções que
operam sobre eles.

A **Programação Orientada a Objetos (POO)** resolve isso: ela junta, dentro de uma
**classe**, os *dados* (atributos) e os *comportamentos* (métodos).

Neste notebook vou:

- entender o que é uma classe e um objeto;
- criar a classe `Aluno`;
- criar a classe `SistemaAlunos` (que gerencia vários alunos);
- conectar com a validação do Notebook 04;
- terminar usando o módulo final `sistema_alunos.py`.

## 1. A primeira classe

Uma **classe** é como uma "planta" (um molde). A partir dela eu crio **objetos**
(as casas construídas a partir da planta).

- `__init__` é o método chamado quando crio o objeto;
- `self` representa o próprio objeto e guarda seus atributos.

In [ ]:
class Pessoa:
    def __init__(self, nome):
        self.nome = nome  # atributo guardado dentro do objeto

    def apresentar(self):
        return f"Olá, eu sou {self.nome}."


p = Pessoa("David")     # criando um objeto (uma instância)
print(p.nome)
print(p.apresentar())

## 2. A classe Aluno

Agora transformo o dicionário de aluno em uma classe. Repare que o cálculo do
valor mensal vira um **método** — o comportamento mora junto com os dados.

In [ ]:
SEMANAS_NO_MES = 4


class Aluno:
    def __init__(self, nome, serie, disciplina, valor_aula, aulas_por_semana):
        self.nome = nome
        self.serie = serie
        self.disciplina = disciplina
        self.valor_aula = valor_aula
        self.aulas_por_semana = aulas_por_semana

    def valor_mensal(self):
        return self.valor_aula * self.aulas_por_semana * SEMANAS_NO_MES

In [ ]:
ana = Aluno("Ana", "9º ano", "Matemática", 130.0, 2)

print("Nome:", ana.nome)
print("Valor mensal: R$", ana.valor_mensal())

## 3. O método especial `__str__`

Quando dou `print` em um objeto, o Python chama o método `__str__`. Definindo ele,
controlo como o aluno aparece na tela — bem mais legível.

In [ ]:
class Aluno:
    def __init__(self, nome, serie, disciplina, valor_aula, aulas_por_semana):
        self.nome = nome
        self.serie = serie
        self.disciplina = disciplina
        self.valor_aula = valor_aula
        self.aulas_por_semana = aulas_por_semana

    def valor_mensal(self):
        return self.valor_aula * self.aulas_por_semana * SEMANAS_NO_MES

    def __str__(self):
        return (
            f"{self.nome} — {self.serie} | {self.disciplina} | "
            f"R$ {self.valor_aula:.2f}/aula | mensal: R$ {self.valor_mensal():.2f}"
        )


ana = Aluno("Ana", "9º ano", "Matemática", 130.0, 2)
print(ana)   # agora usa o __str__

## 4. Validação dentro da classe

Juntando com o que aprendi no Notebook 04: a própria classe pode recusar dados
inválidos no momento da criação, usando `raise`. Assim é **impossível** existir um
`Aluno` com valor de aula negativo.

In [ ]:
class Aluno:
    def __init__(self, nome, serie, disciplina, valor_aula, aulas_por_semana):
        if not nome.strip():
            raise ValueError("O nome não pode ficar vazio.")
        if float(valor_aula) <= 0:
            raise ValueError("O valor da aula precisa ser maior que zero.")
        if int(aulas_por_semana) <= 0:
            raise ValueError("As aulas por semana precisam ser maior que zero.")

        self.nome = nome.strip()
        self.serie = serie.strip()
        self.disciplina = disciplina.strip()
        self.valor_aula = float(valor_aula)
        self.aulas_por_semana = int(aulas_por_semana)

    def valor_mensal(self):
        return self.valor_aula * self.aulas_por_semana * SEMANAS_NO_MES

    def __str__(self):
        return (
            f"{self.nome} — {self.serie} | {self.disciplina} | "
            f"R$ {self.valor_aula:.2f}/aula | mensal: R$ {self.valor_mensal():.2f}"
        )


# válido
print(Aluno("Ana", "9º ano", "Matemática", 130.0, 2))

# inválido
try:
    Aluno("Erro", "9º ano", "Matemática", -10, 2)
except ValueError as erro:
    print("Recusado:", erro)

## 5. A classe SistemaAlunos

Agora crio uma segunda classe, responsável por **gerenciar** vários alunos. Ela
guarda a lista e oferece métodos para cadastrar, listar e gerar relatório.

Isso é a ideia de **responsabilidades separadas**: `Aluno` cuida de um aluno;
`SistemaAlunos` cuida da coleção.

In [ ]:
class SistemaAlunos:
    def __init__(self):
        self.alunos = []

    def cadastrar(self, nome, serie, disciplina, valor_aula, aulas_por_semana):
        aluno = Aluno(nome, serie, disciplina, valor_aula, aulas_por_semana)
        self.alunos.append(aluno)
        return aluno

    def total_mensal(self):
        return sum(aluno.valor_mensal() for aluno in self.alunos)

    def relatorio(self):
        print("RELATÓRIO DE ALUNOS")
        print("=" * 40)
        for aluno in self.alunos:
            print(aluno)
        print("-" * 40)
        print(f"Total mensal estimado: R$ {self.total_mensal():.2f}")

In [ ]:
sistema = SistemaAlunos()
sistema.cadastrar("Ana", "9º ano", "Matemática", 130.0, 2)
sistema.cadastrar("João", "8º ano", "Ciências", 120.0, 1)
sistema.cadastrar("Maria", "1ª série EM", "Física", 150.0, 1)

sistema.relatorio()

## 6. Salvando objetos em JSON

Um detalhe importante: o `json` não sabe salvar um objeto `Aluno` diretamente.
A solução é dar à classe um método `to_dict()` (objeto → dicionário) e um
`from_dict()` (dicionário → objeto). Foi exatamente assim que fiz no módulo final.

In [ ]:
import json

# transformando objetos em dicionários para salvar
dados = [
    {
        "nome": a.nome,
        "serie": a.serie,
        "disciplina": a.disciplina,
        "valor_aula": a.valor_aula,
        "aulas_por_semana": a.aulas_por_semana,
    }
    for a in sistema.alunos
]

print(json.dumps(dados, ensure_ascii=False, indent=2))

## 7. Usando o módulo final `sistema_alunos.py`

Todo esse código — com validação, persistência e relatório — já está organizado no
arquivo **`sistema_alunos.py`**, na mesma pasta. Em um projeto real, a gente importa
o módulo em vez de reescrever as classes.

> Dica: o módulo também tem testes automatizados em `test_sistema_alunos.py`.
> Rode no terminal, dentro desta pasta: `python3 -m unittest -v`

In [ ]:
from sistema_alunos import SistemaAlunos

s = SistemaAlunos()          # usa "alunos.json" por padrão
s.carregar()                 # lê os alunos salvos no arquivo
print(f"{len(s.alunos)} aluno(s) carregado(s).\n")
print(s.relatorio())         # no módulo, relatorio() devolve texto

## Conclusão do Notebook 05

Neste notebook dei o salto da programação com funções soltas para a
**Orientação a Objetos**. Aprendi a:

- criar **classes** e **objetos** (`__init__`, `self`);
- transformar o aluno (que era um dicionário) na classe `Aluno`, com o método
  `valor_mensal()` e o `__str__`;
- colocar a **validação dentro da classe** (impossível criar aluno inválido);
- criar a classe `SistemaAlunos` para gerenciar a coleção;
- converter objetos para JSON com `to_dict()` / `from_dict()`;
- **importar e reutilizar** o módulo `sistema_alunos.py`.

### O sistema agora está completo

O projeto saiu de um simples script de revisão (Notebook 01) e virou um sistema
organizado, validado, testado e dividido em classes.

### Próximos passos (rumo à Engenharia de Software)

- trocar o arquivo JSON por um **banco de dados SQL** (SQLite);
- transformar o sistema em uma **API** (FastAPI / Flask);
- criar uma **interface gráfica** ou web;
- aprofundar testes e versionamento com **Git/GitHub**.